# Phase 2A: 민감도 분석 및 SHAP 보완

**입력:** `phase1c_output.zip`, `phase1c2_output.zip` 두 파일. **출력:** `phase2a_output.zip`.

검증셋(8월)에서만 조기종료하고 테스트셋(9월)은 마지막에 평가. (1) 학습셋에서 시즌 초 승률 0.5 대체가 있었던 154경기를 제외한 C1 재학습, (2) 선형결합 변수들을 제거한 reduced-feature C1 재학습, (3) reduced-feature C1에 대한 Gradient SHAP. 기존 2025 9월 테스트를 이미 여러 차례 확인하였으므로 이 분석은 *사후 민감도*이며 새 독립 테스트로 주장하지 않습니다.

In [ ]:
!pip -q install tensorflow shap scikit-learn pandas matplotlib
import os,io,json,zipfile,random
import numpy as np,pandas as pd,tensorflow as tf,shap,matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,roc_auc_score,brier_score_loss,log_loss
from google.colab import files
SEED=2026
np.random.seed(SEED); random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print('phase1c_output.zip과 phase1c2_output.zip을 함께 업로드해 주세요.')
u=files.upload(); assert {'phase1c_output.zip','phase1c2_output.zip'}.issubset(u), '두 zip 파일의 이름을 확인하세요.'
def get(z,f):return zipfile.ZipFile(io.BytesIO(u[z])).read(f)
df=pd.read_csv(io.BytesIO(get('phase1c_output.zip','pregame_features_cleaned_2430.csv')))
export=json.loads(get('phase1c2_output.zip','selected_pregame_export.json'))
old=pd.read_csv(io.BytesIO(get('phase1c2_output.zip','selected_test_predictions.csv')))
ALL=export['features']
assert df.gamePk.nunique()==2430 and len(ALL)==22 and set(df.split)=={'train','validation','test'}
print('Split',df.split.value_counts().to_dict())
print('시즌 초 대체 경기:',df.loc[df.home_winpct_imputed|df.away_winpct_imputed,'split'].value_counts().to_dict())

## 1. C1 구조 재학습: 전체 Train vs 초기 대체 경기 제외 Train

원래 C1과 같은 구조(128-64-32, Dropout 0.15, L2 0.0001). 초기 제외의 효과를 분리하기 위해 별도 StandardScaler를 *각 학습셋에만* 적합합니다. 종료 에폭 등 시드에 따른 차이는 함께 기록합니다.

In [ ]:
def build(in_dim):
    m=tf.keras.Sequential([tf.keras.Input(shape=(in_dim,)),
        tf.keras.layers.Dense(128,activation='relu',kernel_regularizer=tf.keras.regularizers.l2(.0001)),
        tf.keras.layers.Dropout(.15),tf.keras.layers.Dense(64,activation='relu',kernel_regularizer=tf.keras.regularizers.l2(.0001)),
        tf.keras.layers.Dropout(.15),tf.keras.layers.Dense(32,activation='relu',kernel_regularizer=tf.keras.regularizers.l2(.0001)),
        tf.keras.layers.Dense(1,activation='sigmoid')])
    m.compile(optimizer=tf.keras.optimizers.Adam(.001),loss='binary_crossentropy')
    return m

def metrics(y,p):
    p=np.clip(np.asarray(p).reshape(-1),1e-8,1-1e-8)
    return dict(accuracy=float(accuracy_score(y,p>=.5)),auc=float(roc_auc_score(y,p)),brier=float(brier_score_loss(y,p)),log_loss=float(log_loss(y,p)))

REDUCED=[x for x in ALL if x not in ['win_pct_diff','home_run_diff','away_run_diff','run_diff_diff','form_diff','sp_era_diff','sp_whip_diff','sp_k9_diff']]
# Keep fundamental per-team components, remove algebraically derived duplicates.
print('Reduced feature list',len(REDUCED),REDUCED)
config=[('C1_refit_all_train',ALL,False),('C1_early154_removed',ALL,True),('C1_reduced14',REDUCED,False)]
results=[]; modelpacks={}; prediction_rows=[]
for label,features,exclude in config:
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
    tr=df[df.split=='train'].copy()
    if exclude:tr=tr.loc[~(tr.home_winpct_imputed|tr.away_winpct_imputed)].copy()
    va=df[df.split=='validation'];te=df[df.split=='test']
    sc=StandardScaler().fit(tr[features]);
    Xtr=sc.transform(tr[features]);Xva=sc.transform(va[features]);Xte=sc.transform(te[features])
    m=build(len(features))
    h=m.fit(Xtr,tr.target.to_numpy(),validation_data=(Xva,va.target.to_numpy()),epochs=150,batch_size=128,verbose=0,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=12,restore_best_weights=True)])
    p=m.predict(Xte,verbose=0).ravel()
    row={'experiment':label,'train_games':len(tr),'n_features':len(features),'best_epoch':int(np.argmin(h.history['val_loss'])+1),'epochs_run':len(h.history['loss']),**metrics(te.target,p)}
    results.append(row);modelpacks[label]=(m,sc,features,tr,va,te)
    prediction_rows.append(pd.DataFrame({'gamePk':te.gamePk,'date':te.date,'target':te.target,'experiment':label,'prob':p}))
    print(row)
pd.DataFrame(results).to_csv('phase2a_sensitivity_metrics.csv',index=False)
pd.concat(prediction_rows,ignore_index=True).to_csv('phase2a_sensitivity_predictions.csv',index=False)
print('기존 고정 C1 (실제 이전 실험):',metrics(old.target,old.prob_selected_dnn))

## 2. Reduced-feature C1 SHAP

공선성을 줄인 14개 원 변수로 재학습한 *별도 민감도 모델*을 설명합니다. 원래 22변수 C1의 설명을 대체하는 것으로 자동 간주해서는 안 됩니다. SHAP 결과의 표본과 배경 데이터 크기를 명시합니다.

In [ ]:
m,sc,features,tr,va,te=modelpacks['C1_reduced14']
rng=np.random.default_rng(SEED)
bg=tr.iloc[rng.choice(len(tr),min(100,len(tr)),replace=False)]
ex=te.iloc[rng.choice(len(te),min(150,len(te)),replace=False)]
xb=sc.transform(bg[features]).astype(np.float32)
xe=sc.transform(ex[features]).astype(np.float32)
explainer=shap.GradientExplainer(m,xb)
values=np.asarray(explainer.shap_values(xe,nsamples=100,rseed=SEED))
if values.ndim==3:values=values[:,:,0]
imp=pd.DataFrame({'feature':features,'mean_abs_shap':np.mean(np.abs(values),axis=0),'mean_signed_shap':values.mean(axis=0)})
imp=imp.sort_values('mean_abs_shap',ascending=False)
imp.to_csv('phase2a_reduced14_shap.csv',index=False)
print(imp.to_string(index=False))
plt.figure(figsize=(9,5));plt.barh(imp.feature[::-1],imp.mean_abs_shap[::-1]);plt.xlabel('Mean |SHAP value|');plt.tight_layout();plt.savefig('phase2a_reduced14_shap.png',dpi=160);plt.show()

## 3. 재현 기록 및 결과 ZIP

In [ ]:
audit={'seed':SEED,'original_2025_split':'train through 2025-08-03, validation Aug 04-31, test Sep 01-28',
 'early_imputed_total':int((df.home_winpct_imputed|df.away_winpct_imputed).sum()),
 'early_imputed_test':int(((df.home_winpct_imputed|df.away_winpct_imputed)&(df.split=='test')).sum()),
 'early_imputed_train':int(((df.home_winpct_imputed|df.away_winpct_imputed)&(df.split=='train')).sum()),
 'shap_explainer':'GradientExplainer','shap_background':len(bg),'shap_explanation_n':len(ex),
 'reduced_features':features,'note':'Repeated use of Sep 2025 test makes this sensitivity/exploratory analysis, not a new pristine holdout'}
with open('phase2a_sensitivity_audit.json','w') as f:json.dump(audit,f,indent=2)
outputs=['phase2a_sensitivity_metrics.csv','phase2a_sensitivity_predictions.csv','phase2a_reduced14_shap.csv','phase2a_reduced14_shap.png','phase2a_sensitivity_audit.json']
with zipfile.ZipFile('phase2a_output.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in outputs:z.write(f)
files.download('phase2a_output.zip')